# 🔍 Internship Certificate Fraud Detection
## Transfer Learning with EfficientNetB0

This notebook trains a deep learning model to classify internship certificates as **REAL** or **FAKE** using transfer learning.

### Steps:
1. Upload Kaggle API key & download dataset
2. Load and augment images
3. Train with 2-phase transfer learning (frozen → fine-tune)
4. Evaluate with 5-fold cross-validation
5. Test on user-uploaded images

## Step 1: Setup Kaggle API & Download Dataset

In [ ]:
# Upload your kaggle.json API key file
from google.colab import files
print("Please upload your kaggle.json file.")
print("You can get it from: Kaggle → Account → Create New API Token")
uploaded = files.upload()

In [ ]:
# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle API key configured.")

In [ ]:
# Download the dataset
!kaggle datasets download -d pranavchaudhari30/ai-based-document-fraud-detection
!unzip -o ai-based-document-fraud-detection.zip -d /content/
print("\n✅ Dataset downloaded and extracted.")

In [ ]:
# Verify the dataset folders exist
import os

# Check what was extracted
print("Contents of /content/:")
for item in sorted(os.listdir('/content/')):
    full_path = os.path.join('/content/', item)
    if os.path.isdir(full_path):
        num_files = len(os.listdir(full_path))
        print(f"  📁 {item}/ ({num_files} files)")
    else:
        print(f"  📄 {item}")

# Verify the expected directories
fake_dir = "/content/fake internship certificate"
real_dir = "/content/real internship certificate"

if os.path.isdir(fake_dir):
    print(f"\n✅ Fake certificates folder found: {len(os.listdir(fake_dir))} files")
else:
    print(f"\n❌ Fake certificates folder NOT found at: {fake_dir}")
    print("   Looking for alternative paths...")
    for item in os.listdir('/content/'):
        if 'fake' in item.lower():
            print(f"   Found: /content/{item}")

if os.path.isdir(real_dir):
    print(f"✅ Real certificates folder found: {len(os.listdir(real_dir))} files")
else:
    print(f"❌ Real certificates folder NOT found at: {real_dir}")
    print("   Looking for alternative paths...")
    for item in os.listdir('/content/'):
        if 'real' in item.lower():
            print(f"   Found: /content/{item}")

## Step 2: Imports & Configuration

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                        ModelCheckpoint, TensorBoard)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shutil
import glob
import cv2
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
CONFIG = {
    # ── Paths ──────────────────────────────────
    "fake_dir":  "/content/fake internship certificate",
    "real_dir":  "/content/real internship certificate",

    # ── Model ──────────────────────────────────
    "img_size":     (224, 224),
    "batch_size":   4,            # small batch for tiny dataset
    "epochs_frozen": 40,          # train head only
    "epochs_finetune": 60,        # fine-tune top layers
    "lr_frozen":    1e-3,
    "lr_finetune":  1e-5,
    "dropout":      0.4,
    "l2_reg":       1e-4,

    # ── Data Split ─────────────────────────────
    "val_split":    0.2,
    "n_folds":      5,            # 5-fold cross-validation

    # ── Augmentation ───────────────────────────
    "aug_factor":   25,           # generate 25× synthetic images per original

    # ── Output ─────────────────────────────────
    "output_dir":   "fraud_detection_output",
    "model_name":   "best_fraud_model.keras",
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
CLASSES = ["real", "fake"]   # 0 = real, 1 = fake
print("✅ Configuration set.")

## Step 3: Data Loading & Augmentation

In [ ]:
def load_images_from_folder(folder: str, img_size=(224, 224)):
    """Load all images from a folder, return (images_np, paths)."""
    images, paths = [], []
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
    for ext in exts:
        for p in sorted(glob.glob(os.path.join(folder, ext))):
            img = cv2.imread(p)
            if img is None:
                print(f"  ⚠ Could not read {p}, skipping.")
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, img_size)
            images.append(img)
            paths.append(p)
    return np.array(images, dtype=np.float32), paths


def load_dataset(config):
    """Load real + fake images, return (X, y, paths)."""
    print("📂 Loading dataset …")

    # ⚠️ IMPORTANT: Load real images from real_dir, fake from fake_dir
    X_real, p_real = load_images_from_folder(config["real_dir"], config["img_size"])
    X_fake, p_fake = load_images_from_folder(config["fake_dir"], config["img_size"])

    print(f"  Real certificates : {len(X_real)}")
    print(f"  Fake certificates : {len(X_fake)}")

    if len(X_real) == 0 or len(X_fake) == 0:
        raise ValueError(
            f"Dataset is empty! Real: {len(X_real)}, Fake: {len(X_fake)}.\n"
            f"Check that these paths exist and contain images:\n"
            f"  real_dir: {config['real_dir']}\n"
            f"  fake_dir: {config['fake_dir']}"
        )

    X = np.concatenate([X_real, X_fake], axis=0)
    y = np.array([0] * len(X_real) + [1] * len(X_fake), dtype=np.int32)
    paths = p_real + p_fake

    print(f"  Total images      : {len(X)}")
    print(f"  Label distribution: real={np.sum(y==0)}, fake={np.sum(y==1)}")
    return X, y, paths


# Test loading
X, y, paths = load_dataset(CONFIG)

In [ ]:
def augment_offline(X: np.ndarray, y: np.ndarray, factor: int = 25):
    """
    Offline augmentation: expand tiny dataset N× before training.
    Adds rotation, flips, brightness, zoom, shear, channel shifts.
    """
    aug = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.15,
        height_shift_range=0.15,
        shear_range=0.15,
        zoom_range=0.2,
        horizontal_flip=True,
        vertical_flip=False,
        brightness_range=[0.6, 1.4],
        channel_shift_range=25,
        fill_mode="reflect",
    )

    X_aug, y_aug = [X.copy()], [y.copy()]

    for i in range(factor - 1):
        batch = np.zeros_like(X)
        for j, img in enumerate(X):
            sample = img[np.newaxis]
            gen = aug.flow(sample, batch_size=1)
            batch[j] = next(gen)[0]
        X_aug.append(batch)
        y_aug.append(y.copy())
        if (i + 1) % 5 == 0:
            print(f"    Augmentation progress: {i + 2}/{factor}")

    X_out = np.concatenate(X_aug, axis=0)
    y_out = np.concatenate(y_aug, axis=0)

    # Clip pixel values to valid [0, 255] range after augmentation
    # (brightness_range and channel_shift_range can push values outside this range)
    X_out = np.clip(X_out, 0, 255)

    print(f"  After augmentation : {len(X_out)} images ({factor}×)")
    return X_out, y_out


# Online augmentation layer inside the model
def build_augmentation_layer():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.2),
    ], name="online_augmentation")

print("✅ Augmentation functions defined.")

## Step 4: Model Architecture

In [ ]:
def build_model(config, trainable_base=False):
    """
    EfficientNetB0 transfer learning:
      • Phase 1  – frozen base, train classification head
      • Phase 2  – unfreeze top N layers for fine-tuning

    NOTE: EfficientNetB0 in TF 2.x includes built-in Rescaling
    and Normalization layers, so we pass raw [0, 255] pixels.
    Do NOT apply preprocess_input externally — that would
    double-normalize the data.
    """
    inputs = keras.Input(shape=(*config["img_size"], 3), name="input_image")

    # Online augmentation (only active during training, auto-disabled during predict)
    x = build_augmentation_layer()(inputs)

    # Pre-trained backbone (includes internal preprocessing)
    base = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=x,
    )
    base.trainable = trainable_base

    x = base.output

    # Classification head
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=keras.regularizers.l2(config["l2_reg"]),
        name="dense_256",
    )(x)
    x = layers.Dropout(config["dropout"])(x)
    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=keras.regularizers.l2(config["l2_reg"]),
        name="dense_128",
    )(x)
    x = layers.Dropout(config["dropout"] * 0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = Model(inputs=inputs, outputs=outputs, name="FraudDetector")
    return model, base


def unfreeze_top_layers(model, base_model, n_layers=30):
    """Unfreeze the top N layers of the backbone for fine-tuning."""
    base_model.trainable = True
    for layer in base_model.layers[:-n_layers]:
        layer.trainable = False
    trainable_count = sum(1 for l in base_model.layers if l.trainable)
    print(f"  Backbone trainable layers: {trainable_count} / {len(base_model.layers)}")

print("✅ Model architecture defined.")

## Step 5: Training Pipeline

In [ ]:
def get_callbacks(config, fold=None):
    suffix = f"_fold{fold}" if fold is not None else ""
    ckpt_path = os.path.join(config["output_dir"], f"ckpt{suffix}.keras")
    return [
        ModelCheckpoint(ckpt_path, monitor="val_auc", mode="max",
                        save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_auc", mode="max",
                      patience=15, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=7, min_lr=1e-7, verbose=1),
    ]


def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )


def train_single_split(X_train, y_train, X_val, y_val, config, fold=None):
    """Two-phase training: frozen head → fine-tune."""

    # No external preprocessing needed — EfficientNetB0 includes
    # built-in Rescaling + Normalization layers that handle [0,255] input

    # ── Phase 1: Train head only ───────────────
    print("\n  Phase 1: Training classification head (frozen backbone) …")
    model, base = build_model(config, trainable_base=False)
    compile_model(model, config["lr_frozen"])

    callbacks = get_callbacks(config, fold)
    h1 = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=config["epochs_frozen"],
        batch_size=config["batch_size"],
        class_weight={0: 1.0, 1: 1.0},
        callbacks=callbacks,
        verbose=1,
    )

    # ── Phase 2: Fine-tune top layers ──────────
    print("\n  Phase 2: Fine-tuning top backbone layers …")
    unfreeze_top_layers(model, base, n_layers=30)
    compile_model(model, config["lr_finetune"])

    callbacks2 = get_callbacks(config, fold)
    h2 = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=config["epochs_finetune"],
        batch_size=config["batch_size"],
        class_weight={0: 1.0, 1: 1.0},
        callbacks=callbacks2,
        verbose=1,
    )

    return model, h1, h2

print("✅ Training functions defined.")

In [ ]:
def cross_validate(X, y, config):
    skf = StratifiedKFold(n_splits=config["n_folds"], shuffle=True, random_state=42)
    fold_results = []
    best_auc, best_model = 0, None

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n{'='*60}")
        print(f"  FOLD {fold} / {config['n_folds']}")
        print(f"  Train: {len(train_idx)} images  |  Val: {len(val_idx)} images")
        print(f"{'='*60}")

        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        print(f"  Train labels: real={np.sum(y_tr==0)}, fake={np.sum(y_tr==1)}")
        print(f"  Val   labels: real={np.sum(y_val==0)}, fake={np.sum(y_val==1)}")

        # Offline augmentation on training split only
        X_tr_aug, y_tr_aug = augment_offline(X_tr, y_tr, factor=config["aug_factor"])

        model, h1, h2 = train_single_split(X_tr_aug, y_tr_aug,
                                           X_val, y_val,
                                           config, fold=fold)

        # Evaluate — pass raw [0,255] pixels, model handles preprocessing internally
        y_prob = model.predict(X_val.astype(np.float32), verbose=0).ravel()
        y_pred = (y_prob >= 0.5).astype(int)
        try:
            auc = roc_auc_score(y_val, y_prob)
        except Exception:
            auc = 0.0

        acc = np.mean(y_pred == y_val)
        print(f"\n  ✔ Fold {fold} → Accuracy: {acc:.3f}  AUC: {auc:.3f}")
        fold_results.append({"fold": fold, "acc": acc, "auc": auc,
                              "h1": h1, "h2": h2, "model": model,
                              "y_val": y_val, "y_pred": y_pred, "y_prob": y_prob})

        if auc > best_auc:
            best_auc = auc
            best_model = model

    return fold_results, best_model

print("✅ Cross-validation function defined.")

## Step 6: Visualization Functions

In [ ]:
def plot_training_history(fold_results, config):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Training History – All Folds", fontsize=14, fontweight="bold")

    metrics = [("accuracy", "Accuracy"), ("loss", "Loss"),
               ("auc", "AUC"), ("precision", "Precision")]

    for ax, (metric, title) in zip(axes.ravel(), metrics):
        for r in fold_results:
            h1_vals = r["h1"].history.get(metric, [])
            h2_vals = r["h2"].history.get(metric, [])
            combined = h1_vals + h2_vals
            val_h1  = r["h1"].history.get(f"val_{metric}", [])
            val_h2  = r["h2"].history.get(f"val_{metric}", [])
            val_combined = val_h1 + val_h2
            epochs = range(1, len(combined) + 1)
            ax.plot(epochs, combined,     alpha=0.4, linewidth=1)
            ax.plot(epochs, val_combined, alpha=0.4, linewidth=1, linestyle="--")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(config["output_dir"], "training_history.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {path}")


def plot_confusion_matrix(fold_results, config):
    # Aggregate all validation predictions
    y_true_all = np.concatenate([r["y_val"]  for r in fold_results])
    y_pred_all = np.concatenate([r["y_pred"] for r in fold_results])

    cm = confusion_matrix(y_true_all, y_pred_all)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title("Confusion Matrix (Aggregated CV)", fontweight="bold")
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")
    path = os.path.join(config["output_dir"], "confusion_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {path}")
    print("\n📊 Classification Report (Aggregated CV):")
    print(classification_report(y_true_all, y_pred_all, target_names=CLASSES))


def plot_fold_summary(fold_results, config):
    folds = [r["fold"] for r in fold_results]
    accs  = [r["acc"]  for r in fold_results]
    aucs  = [r["auc"]  for r in fold_results]

    x = np.arange(len(folds))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    bars1 = ax.bar(x - width/2, accs, width, label="Accuracy", color="#4C72B0", alpha=0.85)
    bars2 = ax.bar(x + width/2, aucs, width, label="AUC",      color="#DD8452", alpha=0.85)

    ax.set_xlabel("Fold")
    ax.set_ylabel("Score")
    ax.set_title("Per-Fold Performance", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([f"Fold {f}" for f in folds])
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)

    for bar in bars1 + bars2:
        h = bar.get_height()
        ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

    print(f"\n  Mean Accuracy : {np.mean(accs):.3f} ± {np.std(accs):.3f}")
    print(f"  Mean AUC      : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

    path = os.path.join(config["output_dir"], "fold_summary.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {path}")

print("✅ Plotting functions defined.")

In [ ]:
def predict_image(model, image_path: str, config):
    """
    Run inference on a single certificate image.
    Returns (label, confidence, prob).
    Pass raw [0, 255] pixels — EfficientNetB0 normalizes internally.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, config["img_size"])
    img = img.astype(np.float32)  # [0, 255] range
    img = np.expand_dims(img, axis=0)

    prob = float(model.predict(img, verbose=0)[0][0])
    label = "FAKE" if prob >= 0.5 else "REAL"
    confidence = prob if prob >= 0.5 else 1 - prob
    return label, confidence, prob


def visualize_predictions(model, X, y, paths, config, n=10):
    """Show a grid of predictions on the original images."""
    n = min(n, len(X))
    indices = np.random.choice(len(X), n, replace=False)

    cols = min(5, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    if rows == 1:
        axes = axes[np.newaxis, :] if cols > 1 else np.array([[axes]])
    fig.suptitle("Model Predictions on Original Images", fontsize=13, fontweight="bold")

    for i, (ax, idx) in enumerate(zip(axes.ravel(), indices)):
        img   = X[idx].astype(np.uint8)
        label = y[idx]
        prob  = float(model.predict(img[np.newaxis].astype(np.float32), verbose=0)[0][0])
        pred  = int(prob >= 0.5)
        correct = pred == label

        color = "#2ecc71" if correct else "#e74c3c"
        ax.imshow(img)
        ax.set_title(
            f"True: {CLASSES[label]}\nPred: {CLASSES[pred]} ({prob:.2f})",
            color=color, fontsize=9, fontweight="bold"
        )
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)
            spine.set_visible(True)

    # Hide unused axes
    for j in range(i + 1, rows * cols):
        axes.ravel()[j].axis("off")

    plt.tight_layout()
    path = os.path.join(config["output_dir"], "predictions_grid.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {path}")

print("✅ Prediction functions defined.")

## Step 7: Train the Model 🚀

In [ ]:
def main():
    print("=" * 60)
    print("  Internship Certificate Fraud Detection")
    print("  Transfer Learning — EfficientNetB0")
    print("=" * 60)

    # ── GPU info ──────────────────────────────
    gpus = tf.config.list_physical_devices("GPU")
    print(f"\n  GPUs available: {len(gpus)}")
    if gpus:
        try:
            tf.config.experimental.set_memory_growth(gpus[0], True)
        except RuntimeError:
            pass  # Already set

    # ── Load data ─────────────────────────────
    X, y, paths = load_dataset(CONFIG)

    # ── Cross-validation ──────────────────────
    print("\n🔄 Starting 5-Fold Cross-Validation …")
    fold_results, best_model = cross_validate(X, y, CONFIG)

    # ── Save best model ───────────────────────
    model_path = os.path.join(CONFIG["output_dir"], CONFIG["model_name"])
    best_model.save(model_path)
    print(f"\n✅ Best model saved → {model_path}")

    # ── Plots ─────────────────────────────────
    print("\n📈 Generating plots …")
    plot_training_history(fold_results, CONFIG)
    plot_confusion_matrix(fold_results, CONFIG)
    plot_fold_summary(fold_results, CONFIG)
    visualize_predictions(best_model, X, y, paths, CONFIG)

    # ── Example single-image prediction ───────
    print("\n🔍 Example single-image inference:")
    for p in paths[:3]:
        label, conf, prob = predict_image(best_model, p, CONFIG)
        print(f"  {os.path.basename(p):30s} → {label}  (confidence: {conf:.1%})")

    print("\n🎉 Done! All outputs saved to:", CONFIG["output_dir"])
    return best_model, fold_results


best_model, fold_results = main()

## Step 8: Test on Your Own Image 📷

Upload any internship certificate image to test whether it's **REAL** or **FAKE**.

In [ ]:
from google.colab import files

print("📤 Upload an internship certificate image to test:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"\nUploaded image: {image_path}")

In [ ]:
# Load the saved model and make prediction
model_path = os.path.join(CONFIG["output_dir"], CONFIG["model_name"])
loaded_model = tf.keras.models.load_model(model_path)

label, confidence, prob = predict_image(loaded_model, image_path, CONFIG)

# Display the result
print("\n" + "=" * 50)
print(f"  📄 Image: {image_path}")
print(f"  🏷️  Prediction: {label}")
print(f"  📊 Confidence: {confidence:.1%}")
print(f"  📈 Raw probability (fake): {prob:.4f}")
print("=" * 50)

# Show the image
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 6))
plt.imshow(img)
color = "#e74c3c" if label == "FAKE" else "#2ecc71"
plt.title(f"Prediction: {label} ({confidence:.1%})",
          fontsize=16, fontweight="bold", color=color)
plt.axis("off")
plt.show()